Arrays are fast to index but slow to insert and delete — because shifting elements costs O(n). Linked lists flip that trade-off: they sacrifice constant-time index access in exchange for constant-time insertion and deletion anywhere in the list. Understanding *why* requires looking at how linked lists store data — not in a contiguous block, but as individual nodes scattered across the heap, connected by pointers.

## What is a Linked List?

A linked list is a chain of **nodes**. Each node is a small object on the heap containing:
- A **value** — the data being stored
- A **pointer** (reference) to the next node in the chain

The list itself holds a reference to the first node — called the **head**. The last node points to `None` (or `null`), marking the end of the chain.

Because nodes are independent heap objects, they can live at completely different memory addresses. There is no requirement for them to be adjacent. The chain is maintained entirely by the `next` pointers.

![Linked List Nodes](https://raw.githubusercontent.com/schemabotview/dsa/main/img/linked-list-nodes.svg)

## Linked List vs Array

| Operation | Array | Linked List | Why |
|---|---|---|---|
| Access by index | O(1) | O(n) | Array: arithmetic. List: must walk the chain |
| Search | O(n) | O(n) | Both scan sequentially |
| Insert / delete at head | O(n) | O(1) | Array shifts all; list rewires one pointer |
| Insert / delete at tail | O(1)* | O(1)* | *With a tail pointer |
| Insert / delete at middle | O(n) | O(n)** | **O(n) to find position, O(1) to rewire |
| Memory overhead | Low — just values | Higher — each node stores a pointer too ||
| Cache performance | Excellent — contiguous | Poor — nodes scattered across heap ||

> The linked list advantage only materialises when you already *have* a reference to the node you want to insert before/after. If you have to *search* for the position first, the total cost is still O(n).

## Insert and Delete — Pointer Rewiring

The reason insertion and deletion are O(1) *at a known position* is that no elements shift. You only update one or two pointers — regardless of how long the list is.

![Linked List Insert and Delete](https://raw.githubusercontent.com/schemabotview/dsa/main/img/linked-list-insert-delete.svg)

## Singly Linked List — Implementation

### Python

In [ ]:
class Node:
    def __init__(self, val: int):
        self.val  = val
        self.next = None


class LinkedList:
    def __init__(self):
        self.head = None
        self.size = 0

    # ── Build ────────────────────────────────────────────────
    def prepend(self, val: int) -> None:
        """Insert at head — O(1)."""
        node = Node(val)
        node.next = self.head
        self.head  = node
        self.size += 1

    def append(self, val: int) -> None:
        """Insert at tail — O(n) without tail pointer."""
        node = Node(val)
        if not self.head:
            self.head = node
        else:
            curr = self.head
            while curr.next:
                curr = curr.next
            curr.next = node
        self.size += 1

    def insert_after(self, target: int, val: int) -> bool:
        """Insert after the first node with value=target — O(n) to find, O(1) to insert."""
        curr = self.head
        while curr:
            if curr.val == target:
                node      = Node(val)
                node.next = curr.next   # step ①
                curr.next = node        # step ②
                self.size += 1
                return True
            curr = curr.next
        return False

    # ── Remove ───────────────────────────────────────────────
    def remove(self, val: int) -> bool:
        """Delete first node with value=val — O(n)."""
        if not self.head:
            return False
        if self.head.val == val:        # special case: remove head
            self.head  = self.head.next
            self.size -= 1
            return True
        curr = self.head
        while curr.next:
            if curr.next.val == val:
                curr.next  = curr.next.next  # bypass the node
                self.size -= 1
                return True
            curr = curr.next
        return False

    # ── Query ────────────────────────────────────────────────
    def search(self, val: int) -> bool:
        curr = self.head
        while curr:
            if curr.val == val:
                return True
            curr = curr.next
        return False

    def __repr__(self) -> str:
        parts, curr = [], self.head
        while curr:
            parts.append(str(curr.val))
            curr = curr.next
        return ' → '.join(parts) + ' → None'


# Demo
ll = LinkedList()
for v in [10, 30, 50, 20]:
    ll.append(v)

print(ll)                         # 10 → 30 → 50 → 20 → None
ll.prepend(5)
print(ll)                         # 5 → 10 → 30 → 50 → 20 → None
ll.insert_after(30, 99)
print(ll)                         # 5 → 10 → 30 → 99 → 50 → 20 → None
ll.remove(10)
print(ll)                         # 5 → 30 → 99 → 50 → 20 → None
print(ll.search(99))              # True
print(ll.size)                    # 5

### Kotlin

```kotlin
data class Node(val value: Int, var next: Node? = null)

class LinkedList {
    var head: Node? = null
    var size: Int = 0

    fun prepend(value: Int) {
        head = Node(value, head)
        size++
    }

    fun append(value: Int) {
        val node = Node(value)
        if (head == null) { head = node; size++; return }
        var curr = head
        while (curr!!.next != null) curr = curr.next
        curr.next = node
        size++
    }

    fun insertAfter(target: Int, value: Int): Boolean {
        var curr = head
        while (curr != null) {
            if (curr.value == target) {
                curr.next = Node(value, curr.next)
                size++
                return true
            }
            curr = curr.next
        }
        return false
    }

    fun remove(value: Int): Boolean {
        if (head?.value == value) { head = head?.next; size--; return true }
        var curr = head
        while (curr?.next != null) {
            if (curr.next!!.value == value) {
                curr.next = curr.next!!.next
                size--
                return true
            }
            curr = curr.next
        }
        return false
    }

    override fun toString(): String {
        val sb = StringBuilder()
        var curr = head
        while (curr != null) { sb.append("${curr.value} → "); curr = curr.next }
        return sb.append("null").toString()
    }
}
```

## Doubly Linked List

A **doubly linked list** adds a `prev` pointer to every node, pointing to the previous node in the chain. This enables:
- **Backward traversal** — iterate in both directions
- **O(1) delete given the node** — no need to walk the list to find the predecessor; just use `node.prev`

The trade-off is memory: every node now carries two pointers instead of one.

Python's `collections.deque` is implemented as a doubly linked list and gives O(1) append and pop from *both* ends — which makes it the right choice whenever you need a queue or a deque.

### Python

In [ ]:
class DNode:
    def __init__(self, val: int):
        self.val  = val
        self.prev = None
        self.next = None


class DoublyLinkedList:
    def __init__(self):
        # Sentinel nodes — simplify edge cases (no empty-list special-casing)
        self.head       = DNode(0)   # dummy head
        self.tail       = DNode(0)   # dummy tail
        self.head.next  = self.tail
        self.tail.prev  = self.head

    def insert_before_tail(self, val: int) -> DNode:
        """Add just before dummy tail — effectively append — O(1)."""
        node            = DNode(val)
        prev            = self.tail.prev
        prev.next       = node
        node.prev       = prev
        node.next       = self.tail
        self.tail.prev  = node
        return node

    def remove_node(self, node: DNode) -> None:
        """Delete a node given the reference — O(1), no traversal needed."""
        node.prev.next = node.next
        node.next.prev = node.prev

    def __repr__(self) -> str:
        parts, curr = [], self.head.next
        while curr is not self.tail:
            parts.append(str(curr.val))
            curr = curr.next
        return ' ⟺ '.join(parts)


dl = DoublyLinkedList()
n1 = dl.insert_before_tail(10)
n2 = dl.insert_before_tail(30)
n3 = dl.insert_before_tail(50)
print(dl)              # 10 ⟺ 30 ⟺ 50
dl.remove_node(n2)     # O(1) — we have the reference directly
print(dl)              # 10 ⟺ 50

### Kotlin

```kotlin
class DNode(val value: Int, var prev: DNode? = null, var next: DNode? = null)

class DoublyLinkedList {
    private val head = DNode(0)   // sentinel
    private val tail = DNode(0)   // sentinel
    init { head.next = tail; tail.prev = head }

    fun insertBeforeTail(value: Int): DNode {
        val node = DNode(value)
        val prev = tail.prev!!
        prev.next  = node;  node.prev = prev
        node.next  = tail;  tail.prev = node
        return node
    }

    fun removeNode(node: DNode) {
        node.prev!!.next = node.next
        node.next!!.prev = node.prev
    }
}
```

## Two Classic Linked List Algorithms

### Reversing a Linked List

Reverse the chain in-place by walking it once and flipping each `next` pointer to point backward. O(n) time, O(1) space.

### Python

In [ ]:
def reverse(head: Node | None) -> Node | None:
    prev, curr = None, head
    while curr:
        nxt       = curr.next   # save next before overwriting
        curr.next = prev        # flip pointer
        prev      = curr        # advance prev
        curr      = nxt         # advance curr
    return prev                 # prev is the new head


# Build 1 → 2 → 3 → 4 → 5
ll2 = LinkedList()
for v in [1, 2, 3, 4, 5]:
    ll2.append(v)

print(ll2)                           # 1 → 2 → 3 → 4 → 5 → None
ll2.head = reverse(ll2.head)
print(ll2)                           # 5 → 4 → 3 → 2 → 1 → None

### Kotlin

```kotlin
fun reverse(head: Node?): Node? {
    var prev: Node? = null
    var curr = head
    while (curr != null) {
        val nxt   = curr.next
        curr.next = prev
        prev      = curr
        curr      = nxt
    }
    return prev
}
```

### Detecting a Cycle — Floyd's Algorithm

Use two pointers moving at different speeds — a **slow** pointer that advances one node at a time and a **fast** pointer that advances two nodes at a time. If a cycle exists, the fast pointer will eventually lap the slow pointer and they will meet inside the cycle. If there is no cycle, the fast pointer reaches `None`.

This is O(n) time and O(1) space — no visited set needed.

### Python

In [ ]:
def has_cycle(head: Node | None) -> bool:
    slow = fast = head
    while fast and fast.next:
        slow = slow.next          # move 1 step
        fast = fast.next.next     # move 2 steps
        if slow is fast:
            return True           # they met — cycle detected
    return False


# Build a list with a cycle: 1 → 2 → 3 → 4 → 2 (cycle back to node 2)
n1, n2, n3, n4 = Node(1), Node(2), Node(3), Node(4)
n1.next = n2; n2.next = n3; n3.next = n4; n4.next = n2   # cycle

print(has_cycle(n1))    # True

n4.next = None          # break cycle
print(has_cycle(n1))    # False

### Kotlin

```kotlin
fun hasCycle(head: Node?): Boolean {
    var slow = head
    var fast = head
    while (fast != null && fast.next != null) {
        slow = slow!!.next
        fast = fast.next!!.next
        if (slow === fast) return true
    }
    return false
}
```

### Finding the Middle Node

The same fast/slow pointer technique finds the middle of a list in O(n) without knowing its length. When the fast pointer reaches the end, the slow pointer is at the middle.

### Python

In [ ]:
def find_middle(head: Node | None) -> Node | None:
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    return slow   # slow is at the middle when fast reaches the end


ll3 = LinkedList()
for v in [1, 2, 3, 4, 5]:
    ll3.append(v)

print(find_middle(ll3.head).val)   # 3

### Kotlin

```kotlin
fun findMiddle(head: Node?): Node? {
    var slow = head
    var fast = head
    while (fast != null && fast.next != null) {
        slow = slow!!.next
        fast = fast.next!!.next
    }
    return slow
}
```

## When to Use a Linked List

**Choose a linked list when:**
- You need frequent insertions and deletions at the head or tail (queues, LRU cache)
- You have a reference to the node and need O(1) removal (doubly linked list)
- The collection size changes dramatically and unpredictably

**Prefer an array when:**
- You need fast index access (`arr[i]`)
- You iterate sequentially — arrays have far better cache performance because elements are contiguous
- Memory overhead matters — each linked list node stores at least one extra pointer

In practice, Python's `list` (dynamic array) is the right default for most tasks. Use `collections.deque` when you need O(1) operations at both ends. Use a custom linked list only when the problem specifically requires it — most often in interview problems testing pointer manipulation.

## Key Takeaways

- A linked list is a chain of nodes, each holding a value and a pointer to the next node. Nodes are **scattered across the heap** — not contiguous.
- **Index access is O(n)** — you must walk the chain. **Insert/delete at a known position is O(1)** — just rewire one or two pointers.
- A **doubly linked list** adds a `prev` pointer, enabling O(1) deletion given a node reference and backward traversal.
- **Sentinel nodes** (dummy head and tail) eliminate edge cases in doubly linked list operations.
- **Reverse a linked list** with three pointers (`prev`, `curr`, `nxt`) in a single O(n) pass.
- **Floyd's fast/slow pointer** detects cycles in O(n) time and O(1) space. The same technique finds the middle of a list.